# Ecommerce A/B Test Analysis

This notebook pulls the cleaned experiment data from Postgres and walks through an end-to-end analysis of conversion performance, country-level results, session behavior, and statistical significance.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf
from scipy.stats import chi2_contingency
from statsmodels.stats.proportion import confint_proportions_2indep, proportions_ztest

sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

cwd = Path.cwd().resolve()
candidate_roots = [cwd, cwd.parent]
PROJECT_ROOT = next(
    root for root in candidate_roots if (root / "python" / "scripts").exists()
)
SCRIPTS_DIR = PROJECT_ROOT / "python" / "scripts"

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

from pull_from_sql import load_project_data

## Load Data From Postgres

This expects the project `.env` file to point at your Postgres database and the tables `ab_data_clean` and `countries` to already exist.

In [ ]:
datasets = load_project_data()
df_ab = datasets["ab_data_clean"].copy()
df_countries = datasets["countries"].copy()
df_joined = datasets["joined"].copy()

print("ab_data_clean shape:", df_ab.shape)
print("countries shape:", df_countries.shape)
print("joined shape:", df_joined.shape)

In [ ]:
df_joined["session_seconds"] = df_joined["timestamp"].str.split(":").apply(
    lambda parts: float(parts[0]) * 60 + float(parts[1])
)

duplicate_rows = int(df_joined.duplicated("user_id").sum())
duplicate_ids = df_joined.loc[df_joined.duplicated("user_id", keep=False), "user_id"].nunique()

# Sensitivity frame: one row per user to avoid over-counting repeat IDs.
df_users = (
    df_joined.sort_values(["user_id", "timestamp"])
    .drop_duplicates("user_id", keep="first")
    .copy()
)

print("joined duplicate rows:", duplicate_rows)
print("joined duplicate user_ids:", int(duplicate_ids))
print("user-level analysis shape:", df_users.shape)

## Data Quality Check

The cleaned experiment table still contains a very small number of repeated `user_id` values. The impact on results is negligible, but the notebook keeps a user-level version (`df_users`) for the main inference steps.

In [ ]:
group_summary = (
    df_users.groupby("group", dropna=False)
    .agg(
        total_users=("user_id", "count"),
        conversions=("converted", "sum"),
        conversion_rate=("converted", "mean"),
        conversion_rate_pct=("converted", lambda s: s.mean() * 100),
        avg_session_seconds=("session_seconds", "mean"),
        median_session_seconds=("session_seconds", "median"),
    )
    .reset_index()
)

group_summary

In [ ]:
control = df_users.loc[df_users["group"] == "control", "converted"]
treatment = df_users.loc[df_users["group"] == "treatment", "converted"]

count = np.array([treatment.sum(), control.sum()])
nobs = np.array([treatment.shape[0], control.shape[0]])
z_stat, p_value = proportions_ztest(count, nobs)
ci_low, ci_high = confint_proportions_2indep(
    count1=treatment.sum(),
    nobs1=treatment.shape[0],
    count2=control.sum(),
    nobs2=control.shape[0],
    method="newcomb",
)

ab_test_result = pd.DataFrame(
    {
        "metric": [
            "control_conversion_rate_pct",
            "treatment_conversion_rate_pct",
            "treatment_minus_control_pct_points",
            "z_statistic",
            "p_value",
            "95pct_ci_low_pct_points",
            "95pct_ci_high_pct_points",
        ],
        "value": [
            control.mean() * 100,
            treatment.mean() * 100,
            (treatment.mean() - control.mean()) * 100,
            z_stat,
            p_value,
            ci_low * 100,
            ci_high * 100,
        ],
    }
)

ab_test_result

## Conversion By Country

Country-level cuts help us see whether any apparent treatment effect is concentrated in one geography.

In [ ]:
country_summary = (
    df_users.groupby(["country", "group"], dropna=False)
    .agg(
        total_users=("user_id", "count"),
        conversions=("converted", "sum"),
        conversion_rate=("converted", "mean"),
        conversion_rate_pct=("converted", lambda s: s.mean() * 100),
        avg_session_seconds=("session_seconds", "mean"),
    )
    .reset_index()
    .sort_values(["country", "group"])
)

country_summary

In [ ]:
country_pivot = country_summary.pivot(index="country", columns="group", values="conversion_rate_pct")
country_lift = (
    country_pivot.assign(treatment_minus_control_pct_points=country_pivot["treatment"] - country_pivot["control"])
    .reset_index()
    .sort_values("treatment_minus_control_pct_points")
)

country_lift

In [ ]:
country_group_contingency = pd.crosstab(df_users["country"], df_users["group"])
country_conversion_contingency = pd.crosstab(df_users["country"], df_users["converted"])

chi2_country_group, p_country_group, dof_country_group, _ = chi2_contingency(country_group_contingency)
chi2_country_conversion, p_country_conversion, dof_country_conversion, _ = chi2_contingency(country_conversion_contingency)

country_tests = pd.DataFrame(
    {
        "test": ["country_vs_group_balance", "country_vs_conversion_rate"],
        "chi2": [chi2_country_group, chi2_country_conversion],
        "p_value": [p_country_group, p_country_conversion],
        "degrees_of_freedom": [dof_country_group, dof_country_conversion],
    }
)

country_tests

## Logistic Regression

These models check whether the treatment remains meaningful after controlling for geography and whether the treatment effect changes by country.

In [ ]:
model_group = smf.logit("converted ~ C(group)", data=df_users).fit(disp=False)
model_country = smf.logit("converted ~ C(group) + C(country)", data=df_users).fit(disp=False)
model_interaction = smf.logit("converted ~ C(group) * C(country)", data=df_users).fit(disp=False)

regression_summary = pd.DataFrame(
    {
        "model": ["group_only", "group_plus_country", "group_country_interaction"],
        "group_coef": [
            model_group.params.get("C(group)[T.treatment]", np.nan),
            model_country.params.get("C(group)[T.treatment]", np.nan),
            model_interaction.params.get("C(group)[T.treatment]", np.nan),
        ],
        "group_p_value": [
            model_group.pvalues.get("C(group)[T.treatment]", np.nan),
            model_country.pvalues.get("C(group)[T.treatment]", np.nan),
            model_interaction.pvalues.get("C(group)[T.treatment]", np.nan),
        ],
        "aic": [model_group.aic, model_country.aic, model_interaction.aic],
    }
)

regression_summary

## Session Duration

The `timestamp` field is treated as a `mm:ss.s` session-duration measure, following the project SQL analysis.

In [ ]:
session_summary = (
    df_users.groupby(["group", "converted"], dropna=False)
    .agg(
        avg_session_seconds=("session_seconds", "mean"),
        median_session_seconds=("session_seconds", "median"),
        min_session_seconds=("session_seconds", "min"),
        max_session_seconds=("session_seconds", "max"),
    )
    .reset_index()
)

session_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=group_summary, x="group", y="conversion_rate_pct", ax=axes[0])
axes[0].set_title("Overall Conversion Rate by Group")
axes[0].set_ylabel("Conversion Rate (%)")
axes[0].set_xlabel("")

sns.barplot(data=country_summary, x="country", y="conversion_rate_pct", hue="group", ax=axes[1])
axes[1].set_title("Conversion Rate by Country and Group")
axes[1].set_ylabel("Conversion Rate (%)")
axes[1].set_xlabel("")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df_users, x="group", y="session_seconds", ax=axes[0])
axes[0].set_title("Session Duration Distribution by Group")
axes[0].set_ylabel("Session Duration (seconds)")
axes[0].set_xlabel("")

sns.barplot(data=session_summary, x="group", y="avg_session_seconds", hue="converted", ax=axes[1])
axes[1].set_title("Average Session Duration by Group and Conversion")
axes[1].set_ylabel("Average Session Duration (seconds)")
axes[1].set_xlabel("")

plt.tight_layout()
plt.show()

## Takeaways

- The treatment group does **not** beat the control group on conversion.
- The estimated treatment effect is slightly negative, but it is not statistically significant.
- Country-level differences are small and do not meaningfully change the main conclusion.
- Session duration appears broadly similar between groups, so there is no strong engagement-based explanation for a treatment win.